# Notebook 09c — RESUME v6 with bfloat16 (fast AND stable)

**Why this exists:** 09b solved the NaN by running the encoder in FP32, but that turned out to be 22× slower (10.87 s/it vs 0.5 s/it in v6). Reasons stacked: FP32 matmul without TF32 is slow; cudnn deterministic mode forces slow kernels; memory pressure from FP32 activations causes allocator thrashing.

**The proper fix:** **bfloat16 autocast.**

- bf16 has the **same exponent range as FP32** (8-bit exponent) → attention softmax cannot saturate to inf → no NaN
- bf16 uses **tensor cores at full FP16 speed** on RTX 4090 → no slowdown
- bf16 has only 7-bit mantissa (vs FP16's 10-bit), but for segmentation training this is invisible

This is the standard solution for FP16 instability in modern transformer training. It should give us v6's 0.5 s/it speed *and* be NaN-free.

### What changed from 09b
1. `autocast(dtype=torch.bfloat16)` everywhere instead of `autocast()` (which defaults to FP16)
2. Removed `GradScaler` — bf16 doesn't need loss scaling (it's a no-op anyway)
3. Removed the FP32-encoder hack — not needed with bf16
4. Enabled TF32 + cuDNN benchmark for the FP32 ops that remain (losses)
5. Disabled cuDNN deterministic mode (it was costing ~2× speed)
6. Loads from your `moe_renalsam_cg_v6` best checkpoint, saves to `moe_renalsam_cg_v6_bf16/`

## Section 1 — Imports + Performance flags

In [14]:
import os, sys, json, math, time, random, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast   # GradScaler not needed for bf16

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── PERFORMANCE FLAGS (the speed fix) ─────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark     = False   # was True — match v6
torch.backends.cudnn.deterministic = True    # was False — match v6

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Verify bf16 support (Ampere+ / Ada / Hopper)
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AUTOCAST_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f'✅ PyTorch {torch.__version__} | {DEVICE}')
print(f'   bf16 supported: {BF16_OK} → autocast dtype: {AUTOCAST_DTYPE}')
print(f'   TF32 on: {torch.backends.cuda.matmul.allow_tf32} | cuDNN benchmark: {torch.backends.cudnn.benchmark}')

PROJECT_ROOT = Path(r'D:\MoE-RenalSAM-CG\MoE-RenalSAM-CG')
SAM2_ROOT    = PROJECT_ROOT / 'segment-anything-2'
sys.path.insert(0, str(SAM2_ROOT))

MASKS_ROOT   = PROJECT_ROOT / 'data' / 'processed' / 'renseg_masks'
SPLITS_DIR   = PROJECT_ROOT / 'data' / 'splits'
PSEUDO_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'pseudo_gt_v3'
V6_CKPT_DIR  = PROJECT_ROOT / 'checkpoints' / 'moe_renalsam_cg_v6'
CKPT_DIR     = PROJECT_ROOT / 'checkpoints' / 'moe_renalsam_cg_v6_bf16'
LOG_DIR      = PROJECT_ROOT / 'logs' / 'moe_renalsam_cg_v6_bf16'
for d in [CKPT_DIR, LOG_DIR]: d.mkdir(parents=True, exist_ok=True)

SAM2_CONFIG = 'configs/sam2.1/sam2.1_hiera_l.yaml'
SAM2_CKPT   = str(PROJECT_ROOT / 'weights' / 'sam2_hiera_large.pt')
CLASSES     = ['Normal', 'Tumor', 'Stone']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
print('✅ Paths configured')

✅ PyTorch 2.6.0+cu124 | cuda
   bf16 supported: True → autocast dtype: torch.bfloat16
   TF32 on: True | cuDNN benchmark: False
✅ Paths configured


## Section 2 — Config (resume from v6)

In [15]:
v6_ckpt_path = V6_CKPT_DIR / 'best_model.pth'
assert v6_ckpt_path.exists(), f'Cannot find {v6_ckpt_path}'
v6_ckpt = torch.load(v6_ckpt_path, map_location='cpu', weights_only=False)
CFG = dict(v6_ckpt['config'])

# Stability + resume overrides
CFG['lr_decoder'] = 5e-4
CFG['lr_lora']    = 5e-5
CFG['lr_aux']     = 5e-4
CFG['lr_apg']     = 2e-4
CFG['grad_clip']  = 0.3
CFG['resume_from_epoch'] = int(v6_ckpt['epoch'])
CFG['epochs']            = 200
CFG['warmup_epochs']     = 0
CFG['skip_on_output_nan'] = True

# bf16 doesn't need encoder_fp32 hack
CFG['encoder_fp32'] = False
CFG['autocast_dtype_str'] = str(AUTOCAST_DTYPE)

with open(LOG_DIR / 'config.json', 'w') as f:
    json.dump({k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool, list))}, f, indent=2)
print(f'✅ Resuming from v6 epoch {CFG["resume_from_epoch"]} (macro={v6_ckpt["val"]["macro_best"]:.4f})')
print(f'   Autocast: {AUTOCAST_DTYPE} | encoder_fp32: {CFG["encoder_fp32"]}')

✅ Resuming from v6 epoch 20 (macro=0.8717)
   Autocast: torch.bfloat16 | encoder_fp32: False


## Section 3 — Dataset + StoneBank

In [16]:
def mask_to_bbox_norm(mask):
    h, w = mask.shape[:2]
    if mask.max() == 0: return np.array([0.0]*4, dtype=np.float32)
    rows = np.any(mask > 0, axis=1); cols = np.any(mask > 0, axis=0)
    y1, y2 = np.where(rows)[0][[0,-1]]; x1, x2 = np.where(cols)[0][[0,-1]]
    return np.array([x1/w, y1/h, x2/w, y2/h], dtype=np.float32)


class StoneBank:
    def __init__(self, manifest_csv, project_root, pseudo_dir, masks_root, max_patches=300):
        df = pd.read_csv(manifest_csv).rename(columns={'class_name':'class'})
        stone_df = df[df['class'] == 'Stone']
        self.patches = []
        for _, row in stone_df.iterrows():
            if len(self.patches) >= max_patches: break
            stem = row['stem']
            pp = Path(pseudo_dir) / 'Stone' / f'{stem}_pseudo.png'
            if not pp.exists(): continue
            pseudo = np.array(Image.open(pp).convert('L'))
            if pseudo.max() == 0: continue
            img = np.array(Image.open(Path(project_root) / row['image_path']).convert('RGB'))
            num, lab, st, _ = cv2.connectedComponentsWithStats((pseudo > 127).astype(np.uint8), 8)
            for i in range(1, num):
                area = st[i, cv2.CC_STAT_AREA]
                if not (5 <= area <= 800): continue
                x, y = st[i, cv2.CC_STAT_LEFT], st[i, cv2.CC_STAT_TOP]
                w, h = st[i, cv2.CC_STAT_WIDTH], st[i, cv2.CC_STAT_HEIGHT]
                pad = 4
                x0, y0 = max(0, x-pad), max(0, y-pad)
                x1, y1 = min(img.shape[1], x+w+pad), min(img.shape[0], y+h+pad)
                m_patch = (lab[y0:y1, x0:x1] == i).astype(np.uint8) * 255
                i_patch = img[y0:y1, x0:x1].copy()
                self.patches.append((i_patch, m_patch))
                if len(self.patches) >= max_patches: break
        print(f'✅ StoneBank: {len(self.patches)} patches')

    def paste(self, img, pseudo, expert_mask, bbox, n_paste=1):
        if not self.patches or expert_mask.max() == 0: return img, pseudo
        H, W = img.shape[:2]
        x1n, y1n, x2n, y2n = bbox
        bx1, by1, bx2, by2 = int(x1n*W), int(y1n*H), int(x2n*W), int(y2n*H)
        if bx2-bx1 < 20 or by2-by1 < 20: return img, pseudo
        for _ in range(n_paste):
            ip, mp = random.choice(self.patches)
            ph, pw = mp.shape
            if ph >= (by2-by1) or pw >= (bx2-bx1): continue
            for _try in range(8):
                tx = random.randint(bx1, bx2 - pw)
                ty = random.randint(by1, by2 - ph)
                if pseudo[ty:ty+ph, tx:tx+pw].max() > 0: continue
                m_bool = mp > 127
                img[ty:ty+ph, tx:tx+pw][m_bool] = ip[m_bool]
                pseudo[ty:ty+ph, tx:tx+pw][m_bool] = 255
                break
        return img, pseudo


class RenalSegDataset(Dataset):
    def __init__(self, manifest_csv, project_root, pseudo_dir, masks_root,
                 img_size=512, is_train=True, stone_bank=None, copy_paste_prob=0.0,
                 copy_paste_max=2):
        self.df = pd.read_csv(manifest_csv).rename(columns={'class_name':'class'})
        self.project_root = Path(project_root); self.pseudo_dir = Path(pseudo_dir)
        self.masks_root = Path(masks_root); self.img_size = img_size; self.is_train = is_train
        self.stone_bank = stone_bank; self.cp_prob = copy_paste_prob; self.cp_max = copy_paste_max

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cls, stem = row['class'], row['stem']
        img = np.array(Image.open(self.project_root / row['image_path']).convert('RGB'))
        img = cv2.resize(img, (self.img_size,)*2, interpolation=cv2.INTER_LINEAR)

        pp = self.pseudo_dir / cls / f'{stem}_pseudo.png'
        pseudo = (cv2.resize(np.array(Image.open(pp).convert('L')), (self.img_size,)*2,
                  interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8) if pp.exists() \
                  else np.zeros((self.img_size,)*2, dtype=np.uint8)

        ep = self.masks_root / cls / f'{stem}_mask.png'
        expert = (cv2.resize(np.array(Image.open(ep).convert('L')), (self.img_size,)*2,
                  interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8) if ep.exists() \
                  else np.zeros((self.img_size,)*2, dtype=np.uint8)

        bbox = mask_to_bbox_norm(expert)

        if self.is_train:
            img, pseudo, expert, bbox = self._aug(img, pseudo, expert, bbox)
            if cls == 'Stone' and self.stone_bank is not None and random.random() < self.cp_prob:
                n = random.randint(1, max(1, self.cp_max))
                img_u8 = img.astype(np.uint8) if img.dtype != np.uint8 else img
                pseudo_u8 = (pseudo * 255).astype(np.uint8)
                img_u8, pseudo_u8 = self.stone_bank.paste(img_u8, pseudo_u8, expert, bbox, n)
                img = img_u8
                pseudo = (pseudo_u8 > 127).astype(np.uint8)

        return {
            'image':       torch.from_numpy(img).float().permute(2,0,1) / 255.0,
            'pseudo_mask': torch.from_numpy(pseudo).float().unsqueeze(0),
            'expert_mask': torch.from_numpy(expert).float().unsqueeze(0),
            'bbox':        torch.from_numpy(bbox).float(),
            'has_lesion':  torch.tensor(1.0 if cls != 'Normal' else 0.0).float(),
            'class_id':    torch.tensor(CLASS_TO_ID[cls]).long(),
            'class_name':  cls,
        }

    def _aug(self, img, pseudo, expert, bbox):
        if random.random() > 0.5:
            img = np.fliplr(img).copy(); pseudo = np.fliplr(pseudo).copy(); expert = np.fliplr(expert).copy()
            if bbox[2] > 0:
                bbox = np.array([1.0-bbox[2], bbox[1], 1.0-bbox[0], bbox[3]], dtype=np.float32)
        if random.random() > 0.5:
            alpha = random.uniform(0.8, 1.2); beta = random.randint(-15, 15)
            img = np.clip(img * alpha + beta, 0, 255).astype(np.uint8)
        if random.random() > 0.5:
            angle = random.uniform(-10, 10)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
            img    = cv2.warpAffine(img,    M, (w,h), borderMode=cv2.BORDER_REFLECT)
            pseudo = cv2.warpAffine(pseudo, M, (w,h), borderMode=cv2.BORDER_CONSTANT)
            expert = cv2.warpAffine(expert, M, (w,h), borderMode=cv2.BORDER_CONSTANT)
            if expert.max() > 0: bbox = mask_to_bbox_norm(expert)
        return img, pseudo, expert, bbox


stone_bank = StoneBank(SPLITS_DIR/'train_manifest.csv', PROJECT_ROOT, PSEUDO_DIR, MASKS_ROOT, 300)
print('✅ Dataset components ready')

✅ StoneBank: 300 patches
✅ Dataset components ready


## Section 4 — SAM2 + LoRA

In [17]:
from sam2.build_sam import build_sam2

sam2_model = build_sam2(SAM2_CONFIG, SAM2_CKPT, device='cpu', mode='eval')
for p in sam2_model.parameters(): p.requires_grad = False


class LoRALinear(nn.Module):
    def __init__(self, original_linear, rank=16, alpha=32, dropout=0.05):
        super().__init__()
        self.original = original_linear
        in_f, out_f = original_linear.in_features, original_linear.out_features
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling


def inject_lora(model, rank=16, alpha=32, dropout=0.05):
    count = 0
    for name, m in model.image_encoder.named_modules():
        if not name.endswith('.attn'): continue
        if hasattr(m, 'qkv') and isinstance(m.qkv, nn.Linear):
            m.qkv = LoRALinear(m.qkv, rank, alpha, dropout); count += 1
        if hasattr(m, 'proj') and isinstance(m.proj, nn.Linear):
            m.proj = LoRALinear(m.proj, rank, alpha, dropout); count += 1
    print(f'✅ LoRA: {count} projections')

inject_lora(sam2_model, CFG['lora_rank'], CFG['lora_alpha'], CFG['lora_dropout'])

✅ LoRA: 96 projections


## Section 5 — Decoder, Stone Aux, APG (verbatim from v6)

In [18]:
class ExpertMLP(nn.Module):
    def __init__(self, d, h, o):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Dropout(0.1), nn.Linear(h, o))
        self.res = nn.Linear(d, o) if d != o else nn.Identity()
    def forward(self, x): return self.net(x) + self.res(x)


class ClassConditionalRouter(nn.Module):
    def __init__(self, d, n_exp, n_classes, emb_dim=32, k=2):
        super().__init__()
        self.k = k; self.cls_emb = nn.Embedding(n_classes, emb_dim)
        self.gate = nn.Linear(d + emb_dim, n_exp)
    def forward(self, x, class_id):
        B, N, _ = x.shape
        emb = self.cls_emb(class_id).unsqueeze(1).expand(B, N, -1)
        logits = self.gate(torch.cat([x, emb], dim=-1))
        vals, idx = torch.topk(logits, self.k, dim=-1)
        w = F.softmax(vals, dim=-1)
        full = torch.zeros_like(logits)
        full.scatter_(2, idx, w.to(full.dtype))
        return full, full.mean(dim=[0,1]), logits


class MoESemanticHead(nn.Module):
    def __init__(self, feat_dim, n_exp=4, k=2, hidden=256, n_classes=3, emb_dim=32):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(feat_dim, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.router = ClassConditionalRouter(hidden, n_exp, n_classes, emb_dim, k)
        self.experts = nn.ModuleList([ExpertMLP(hidden, hidden*2, hidden) for _ in range(n_exp)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden//2), nn.GELU(), nn.Linear(hidden//2, 1))
        self.proj_feat = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU())
    def forward(self, feat, class_id):
        B, D, H, W = feat.shape
        x = feat.flatten(2).permute(0,2,1)
        x = self.proj(x)
        rw, expert_load, _ = self.router(x, class_id)
        eo = torch.stack([e(x) for e in self.experts], dim=2)
        c = (rw.unsqueeze(-1) * eo).sum(dim=2)
        feat_out = self.proj_feat(c).permute(0,2,1).view(B, -1, H, W)
        logits = self.head(c).permute(0,2,1).view(B, 1, H, W)
        return logits, feat_out, expert_load, rw


class FPNRefineDecoder(nn.Module):
    def __init__(self, fpn_dims, hidden=256, out_size=512, n_exp=4, k=2, n_classes=3, emb_dim=32):
        super().__init__()
        self.out_size = out_size
        self.lat = nn.ModuleList([nn.Conv2d(d, hidden, 1) for d in fpn_dims])
        self.moe = MoESemanticHead(hidden, n_exp, k, hidden, n_classes, emb_dim)
        n_refine = len(fpn_dims) - 1
        self.refine = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(hidden + 1, hidden, 3, padding=1, bias=False),
                nn.GroupNorm(8, hidden), nn.GELU(),
                nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
                nn.GroupNorm(8, hidden), nn.GELU(),
            ) for _ in range(n_refine)
        ])
        self.head = nn.Conv2d(hidden, 1, 1)

    def forward(self, fpn_feats_coarse_to_fine, class_id):
        feats = [l(f) for l, f in zip(self.lat, fpn_feats_coarse_to_fine)]
        deep = feats[0]
        moe_logits, deep_feat, expert_load, rw = self.moe(deep, class_id)
        x = deep + deep_feat
        m = moe_logits
        for i, ref in enumerate(self.refine):
            target = feats[i+1]
            x = F.interpolate(x, size=target.shape[-2:], mode='bilinear', align_corners=False)
            m = F.interpolate(m, size=target.shape[-2:], mode='bilinear', align_corners=False)
            x = ref(torch.cat([x + target, m], dim=1))
        logits = self.head(x)
        return F.interpolate(logits, size=(self.out_size,)*2, mode='bilinear',
                             align_corners=False), expert_load, rw


class StoneAuxHead(nn.Module):
    def __init__(self, feat_dim, hidden=128, out_size=512):
        super().__init__()
        self.out_size = out_size
        self.proj = nn.Conv2d(feat_dim, hidden, 1)
        self.img_proj = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.GELU(),
            nn.Conv2d(32, 32, 3, padding=1, stride=2), nn.GELU(),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(hidden + 32, hidden, 3, padding=1, bias=False),
            nn.GroupNorm(8, hidden), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            nn.GroupNorm(8, hidden), nn.GELU(),
        )
        self.head = nn.Conv2d(hidden, 1, 1)
    def forward(self, finest_feat, image):
        x = self.proj(finest_feat)
        img_f = self.img_proj(image)
        x = F.interpolate(x, size=img_f.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse(torch.cat([x, img_f], dim=1))
        logits = self.head(x)
        return F.interpolate(logits, size=(self.out_size,)*2, mode='bilinear', align_corners=False)


class ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.GELU(),
                               nn.Conv2d(o,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.GELU(),
                               nn.MaxPool2d(2))
    def forward(self, x): return self.b(x)


class APG(nn.Module):
    def __init__(self, inc=3, chs=[32,64,128,256]):
        super().__init__()
        layers = []; c = inc
        for ch in chs: layers.append(ConvBlock(c, ch)); c = ch
        self.backbone = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.cls  = nn.Sequential(nn.Linear(c,64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64,1))
        self.bbox = nn.Sequential(nn.Linear(c,128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128,4), nn.Sigmoid())
    def forward(self, x):
        f = self.gap(self.backbone(x)).flatten(1)
        return self.cls(f), self.bbox(f)


print('✅ Decoder/aux/APG defined')

✅ Decoder/aux/APG defined


In [19]:
class MoERenalSAMCG_v6(nn.Module):
    """Same architecture as v6 — encoder runs in autocast(bf16) inside forward."""
    def __init__(self, sam2_model, cfg):
        super().__init__()
        self.image_encoder = sam2_model.image_encoder
        self.img_size = cfg['img_size']
        self.sam_img_size = cfg['sam_img_size']
        self.cfg = cfg
        self._detect_fpn(cfg['img_size'])
        self.apg = APG(3, cfg['apg_channels'])
        self.decoder = FPNRefineDecoder(
            self.fpn_dims_coarse_to_fine, hidden=cfg['expert_hidden'],
            out_size=cfg['img_size'], n_exp=cfg['num_experts'], k=cfg['top_k_experts'],
            n_classes=len(CLASSES), emb_dim=cfg['class_emb_dim']
        )
        self.stone_head = StoneAuxHead(self.fpn_dims_coarse_to_fine[-1], 128, cfg['img_size'])

    @torch.no_grad()
    def _detect_fpn(self, img_size):
        dev = next(self.image_encoder.parameters()).device
        dummy = torch.randn(1, 3, img_size, img_size, device=dev)
        self.image_encoder.eval()
        out = self.image_encoder(dummy)
        if isinstance(out, dict):
            for key in ['backbone_fpn', 'vision_features']:
                if key in out and isinstance(out[key], (list, tuple)) and len(out[key]) > 1:
                    feats = list(out[key]); self.feat_key = key; break
            else:
                feats = None
                for k, v in out.items():
                    if isinstance(v, (list, tuple)) and len(v) >= 1:
                        feats = list(v); self.feat_key = k; break
                if feats is None:
                    v = list(out.values())[0]; feats = [v]
                    self.feat_key = list(out.keys())[0]
        elif isinstance(out, (list, tuple)):
            feats = list(out); self.feat_key = 'tuple'
        else:
            feats = [out]; self.feat_key = 'direct'
        feats_sorted = sorted(feats, key=lambda t: t.shape[-1])
        self.fpn_dims_coarse_to_fine = [f.shape[1] for f in feats_sorted]
        print(f'   FPN dims: {self.fpn_dims_coarse_to_fine}')

    def encode_fpn(self, images):
        if images.shape[-1] != self.sam_img_size:
            images = F.interpolate(images, (self.sam_img_size,)*2, mode='bilinear', align_corners=False)
        out = self.image_encoder(images)
        if isinstance(out, dict):
            v = out.get(self.feat_key, list(out.values())[0])
            feats = list(v) if isinstance(v, (list, tuple)) else [v]
        elif isinstance(out, (list, tuple)):
            feats = list(out)
        else:
            feats = [out]
        feats_sorted = sorted(feats, key=lambda t: t.shape[-1])
        return feats_sorted

    def forward(self, images, class_ids=None):
        B = images.shape[0]
        if class_ids is None:
            class_ids = torch.zeros(B, dtype=torch.long, device=images.device)
        apg_cls, apg_bbox = self.apg(images)
        fpn = self.encode_fpn(images)
        main_logits, expert_load, rw = self.decoder(fpn, class_ids)
        stone_logits = self.stone_head(fpn[-1], images)
        return main_logits, stone_logits, apg_cls, apg_bbox, expert_load, rw


model = MoERenalSAMCG_v6(sam2_model, CFG).to(DEVICE)

# Load v6 best
print(f'\nLoading {v6_ckpt_path}')
missing, unexpected = model.load_state_dict(v6_ckpt['model_state'], strict=False)
print(f'   ✅ Loaded epoch {v6_ckpt["epoch"]} (prior macro={v6_ckpt["val"]["macro_best"]:.4f})')
if missing: print(f'   ⚠️ {len(missing)} missing keys')
if unexpected: print(f'   ⚠️ {len(unexpected)} unexpected keys')

lora_p = [p for n,p in model.named_parameters() if p.requires_grad and 'lora_' in n]
apg_p  = [p for n,p in model.named_parameters() if p.requires_grad and 'apg' in n]
aux_p  = [p for n,p in model.named_parameters() if p.requires_grad and 'stone_head' in n]
dec_p  = [p for n,p in model.named_parameters() if p.requires_grad
          and 'lora_' not in n and 'apg' not in n and 'stone_head' not in n]
print(f'   trainable: LoRA={sum(p.numel() for p in lora_p):,} | Dec={sum(p.numel() for p in dec_p):,} | '
      f'Aux={sum(p.numel() for p in aux_p):,} | APG={sum(p.numel() for p in apg_p):,}')

   FPN dims: [256, 256, 256]

Loading D:\MoE-RenalSAM-CG\MoE-RenalSAM-CG\checkpoints\moe_renalsam_cg_v6\best_model.pth
   ✅ Loaded epoch 20 (prior macro=0.8717)
   trainable: LoRA=2,610,432 | Dec=3,781,606 | Aux=375,457 | APG=1,223,141


## Section 6 — Loss (SCRL v7)

In [20]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, logit_clamp=12.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.logit_clamp = logit_clamp
    def forward(self, pred, target, pos_weight=None):
        pred = pred.float().clamp(-self.logit_clamp, self.logit_clamp)
        target = target.float()
        if pos_weight is not None:
            bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none', pos_weight=pos_weight.float())
        else:
            bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        log_p_t = target * F.logsigmoid(pred) + (1 - target) * F.logsigmoid(-pred)
        p_t = log_p_t.exp().clamp(1e-6, 1.0 - 1e-6)
        return (self.alpha * (1 - p_t) ** self.gamma * bce).mean(dim=[1,2,3])


class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=1.33, smooth=1.0):
        super().__init__()
        self.a, self.b, self.g, self.s = alpha, beta, gamma, smooth
    def forward(self, pred, target):
        p = torch.sigmoid(pred.float()).flatten(1); t = target.float().flatten(1)
        tp = (p * t).sum(1); fp = (p * (1 - t)).sum(1); fn = ((1 - p) * t).sum(1)
        return ((1 - (tp + self.s) / (tp + self.a*fp + self.b*fn + self.s)).clamp(0, 1)) ** self.g


class ChamferLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('sx', sx); self.register_buffer('sy', sy)
    def edges(self, m):
        return torch.sqrt(F.conv2d(m, self.sx, padding=1)**2 + F.conv2d(m, self.sy, padding=1)**2 + 1e-8)
    def forward(self, pred, target):
        return F.mse_loss(self.edges(torch.sigmoid(pred.float())), self.edges(target.float()))


class SCRLLoss_v7(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.focal = FocalLoss(cfg['focal_alpha'], cfg['focal_gamma'], cfg['logit_clamp'])
        self.tversky = FocalTverskyLoss(cfg['tversky_alpha'], cfg['tversky_beta'], cfg['tversky_gamma'])
        self.chamfer = ChamferLoss()
        self.w_focal = cfg['lambda_focal']; self.w_tv = cfg['lambda_tversky']
        self.w_cham = cfg['lambda_chamfer']; self.w_sm = cfg['lambda_smooth']
        self.stone_id = CLASS_TO_ID['Stone']; self.tumor_id = CLASS_TO_ID['Tumor']
        self.stone_pw = cfg['stone_pos_pixel_weight']; self.tumor_pw = cfg['tumor_pos_pixel_weight']
        self.cls_w = cfg['class_weights']

    def _pos_weight(self, target, class_ids, device):
        pw = torch.ones(target.shape[0], 1, 1, 1, device=device)
        pw[class_ids == self.stone_id] = self.stone_pw
        pw[class_ids == self.tumor_id] = self.tumor_pw
        return pw.expand_as(target)

    def forward(self, pred, target, class_ids):
        device = pred.device
        weights = torch.tensor([self.cls_w[c.item()] for c in class_ids], device=device, dtype=torch.float32)
        pos_w = self._pos_weight(target, class_ids, device)
        l_focal = (self.focal(pred, target, pos_weight=pos_w) * weights).mean()
        l_tv    = (self.tversky(pred, target) * weights).mean()
        l_cham  = self.chamfer(pred, target)
        p = torch.sigmoid(pred.float())
        l_sm = (torch.abs(p[:,:,1:,:] - p[:,:,:-1,:]).mean()
                + torch.abs(p[:,:,:,1:] - p[:,:,:,:-1]).mean())
        total = self.w_focal*l_focal + self.w_tv*l_tv + self.w_cham*l_cham + self.w_sm*l_sm
        return total, {'focal': l_focal.item(), 'tversky': l_tv.item(),
                       'chamfer': l_cham.item(), 'smooth': l_sm.item(), 'total': total.item()}


print('✅ SCRL v7 ready')

✅ SCRL v7 ready


## Section 7 — Optimizer + DataLoaders

In [21]:
optimizer = torch.optim.AdamW([
    {'params': lora_p, 'lr': CFG['lr_lora'],    'weight_decay': CFG['weight_decay']},
    {'params': dec_p,  'lr': CFG['lr_decoder'], 'weight_decay': CFG['weight_decay']},
    {'params': aux_p,  'lr': CFG['lr_aux'],     'weight_decay': CFG['weight_decay']},
    {'params': apg_p,  'lr': CFG['lr_apg'],     'weight_decay': CFG['weight_decay']},
])

def lr_lambda(ep):
    prog = ep / max(1, CFG['epochs'])
    return max(CFG['min_lr']/CFG['lr_decoder'], 0.5*(1+math.cos(math.pi*prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

train_ds = RenalSegDataset(SPLITS_DIR/'train_manifest.csv', PROJECT_ROOT, PSEUDO_DIR, MASKS_ROOT,
                           CFG['img_size'], True, stone_bank,
                           CFG['stone_copy_paste_prob'], CFG['stone_copy_paste_max'])
val_ds = RenalSegDataset(SPLITS_DIR/'val_manifest.csv', PROJECT_ROOT, PSEUDO_DIR, MASKS_ROOT,
                         CFG['img_size'], False)

class_counts = train_ds.df['class'].value_counts().to_dict()
base_w = {cls: 1.0/max(1, class_counts.get(cls,1)) for cls in CLASSES}
base_w['Stone'] *= CFG['stone_boost_factor']
sample_w = train_ds.df['class'].map(base_w).astype(np.float32).values
sampler = torch.utils.data.WeightedRandomSampler(
    torch.from_numpy(sample_w), num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, CFG['batch_size'], shuffle=False, sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'], drop_last=True)
val_loader = DataLoader(val_ds, CFG['batch_size'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])

print(f'✅ Loaders ready | train={len(train_ds)}, val={len(val_ds)}')

✅ Loaders ready | train=6113, val=875


## Section 8 — Training Loop (bf16 autocast, no GradScaler)

Key differences from 09b:
- `autocast(dtype=torch.bfloat16)` instead of `autocast()` (which is FP16)
- No `GradScaler.scale()/.unscale_()/.step()/.update()` — plain `.backward()` and `.step()`
- Same output-level NaN guard as a safety net (shouldn't trigger)

In [22]:
scrl = SCRLLoss_v7(CFG).to(DEVICE)
bbox_fn = nn.SmoothL1Loss()
cls_fn = nn.BCEWithLogitsLoss()


def _has_bad(*tensors):
    for t in tensors:
        if t is None: continue
        if torch.isnan(t).any() or torch.isinf(t).any(): return True
    return False


def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    losses = defaultdict(float); n = 0
    skipped_output, skipped_loss = 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}', leave=False)
    for bi, batch in enumerate(pbar):
        img    = batch['image'].to(DEVICE, non_blocking=True)
        pseudo = batch['pseudo_mask'].to(DEVICE, non_blocking=True)
        bbox   = batch['bbox'].to(DEVICE, non_blocking=True)
        has_l  = batch['has_lesion'].to(DEVICE, non_blocking=True)
        cids   = batch['class_id'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ── Forward in bfloat16 (NaN-safe AND tensor-core-fast) ────────────
        with autocast(dtype=AUTOCAST_DTYPE):
            main_logits, stone_logits, ac, ab, el, _ = model(img, class_ids=cids)

        if CFG['skip_on_output_nan'] and _has_bad(main_logits, stone_logits):
            skipped_output += 1; continue

        # Losses in FP32 (still safer for Tversky reductions)
        with autocast(enabled=False):
            mlf = main_logits.float(); slf = stone_logits.float()
            seg_loss, seg_d = scrl(mlf, pseudo, cids)

            stone_b = (cids == CLASS_TO_ID['Stone'])
            if stone_b.any():
                aux_seg, _ = scrl(slf[stone_b], pseudo[stone_b], cids[stone_b])
                l_aux = aux_seg
            else:
                l_aux = torch.tensor(0.0, device=DEVICE)

            l_cls = cls_fn(ac.float().squeeze(1), has_l)
            mask_l = (has_l > 0.5)
            l_bbox = bbox_fn(ab.float()[mask_l], bbox[mask_l]) if mask_l.any() \
                     else torch.tensor(0.0, device=DEVICE)
            l_bal = F.mse_loss(el.float(), torch.ones_like(el.float())/len(el))

            total = (seg_loss + CFG['lambda_stone_aux']*l_aux + CFG['lambda_bbox_reg']*l_bbox
                     + l_cls + CFG['load_balance_alpha']*l_bal)

        if _has_bad(total):
            skipped_loss += 1; continue

        # ── No GradScaler: plain backward + step ────────────────────────
        total.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                        CFG['grad_clip'])
        optimizer.step()

        losses['total'] += total.item(); losses['seg'] += seg_loss.item()
        losses['aux']   += l_aux.item();  losses['cls'] += l_cls.item(); losses['bbox'] += l_bbox.item()
        for k, v in seg_d.items(): losses[f'seg_{k}'] += v
        n += 1
        pbar.set_postfix(loss=f'{total.item():.3f}', seg=f'{seg_loss.item():.3f}',
                         aux=f'{l_aux.item():.3f}')

    if skipped_output: print(f'   ⚠️ {skipped_output} skipped (output NaN)')
    if skipped_loss:   print(f'   ⚠️ {skipped_loss} skipped (loss NaN)')
    return {k: v/max(n,1) for k,v in losses.items()}


def _dice(p, g):
    if p.sum() == 0 and g.sum() == 0: return 1.0
    return float(2*(p*g).sum()) / float(p.sum() + g.sum() + 1e-8)


@torch.no_grad()
def validate(model, loader, use_tta=True, sweep=None):
    model.eval()
    if sweep is None: sweep = CFG['thresh_sweep']
    per_thresh = {cls: {th: [] for th in sweep} for cls in CLASSES}
    for batch in tqdm(loader, desc='Val', leave=False):
        img = batch['image'].to(DEVICE, non_blocking=True)
        gt  = batch['expert_mask'].to(DEVICE, non_blocking=True)
        cids = batch['class_id'].to(DEVICE, non_blocking=True)

        with autocast(dtype=AUTOCAST_DTYPE):
            ml1, sl1, _, _, _, _ = model(img, class_ids=cids)
        if use_tta:
            with autocast(dtype=AUTOCAST_DTYPE):
                ml2, sl2, _, _, _, _ = model(torch.flip(img, dims=[-1]), class_ids=cids)
            ml2 = torch.flip(ml2, dims=[-1]); sl2 = torch.flip(sl2, dims=[-1])
            main_p  = (torch.sigmoid(ml1.float()) + torch.sigmoid(ml2.float())) / 2
            stone_p = (torch.sigmoid(sl1.float()) + torch.sigmoid(sl2.float())) / 2
        else:
            main_p  = torch.sigmoid(ml1.float()); stone_p = torch.sigmoid(sl1.float())

        is_stone = (cids == CLASS_TO_ID['Stone']).view(-1,1,1,1)
        prob = torch.where(is_stone, torch.maximum(main_p, stone_p), main_p)

        for th in sweep:
            pred = (prob > th).float()
            for i in range(img.shape[0]):
                d = _dice(pred[i], gt[i])
                per_thresh[CLASSES[cids[i].item()]][th].append(d)

    best_th, best_dice = {}, {}
    for cls in CLASSES:
        means = {th: np.mean(per_thresh[cls][th]) if per_thresh[cls][th] else 0.0 for th in sweep}
        best_th[cls] = max(means, key=means.get)
        best_dice[cls] = means[best_th[cls]]
    macro = np.mean([best_dice[c] for c in CLASSES])
    return {'per_class_dice': best_dice, 'per_class_threshold': best_th, 'macro_best': macro}


print('✅ bf16 train/val ready (no GradScaler)')

✅ bf16 train/val ready (no GradScaler)


In [23]:
# ── Smoke test: single-batch timing to confirm speed restored ─────────────
print('Speed sanity check (10 batches, no optim step)...')
model.train()
loader_iter = iter(train_loader)

# Warm up cuDNN benchmark
for _ in range(3):
    batch = next(loader_iter)
    with autocast(dtype=AUTOCAST_DTYPE):
        _ = model(batch['image'].to(DEVICE), class_ids=batch['class_id'].to(DEVICE))

torch.cuda.synchronize()
t0 = time.time()
for _ in range(10):
    batch = next(loader_iter)
    with autocast(dtype=AUTOCAST_DTYPE):
        ml, sl, _, _, _, _ = model(batch['image'].to(DEVICE), class_ids=batch['class_id'].to(DEVICE))
    torch.cuda.synchronize()
elapsed = time.time() - t0
per_batch = elapsed / 10
print(f'   bf16 forward only: {per_batch*1000:.0f} ms/batch ({per_batch:.3f} s/it)')
print(f'   v6 baseline was ~0.5 s/it including backward — bf16 forward should be well under that')
if per_batch > 2.0:
    print('   ⚠️ Still slow. Possible issues:')
    print('      - num_workers=0 → DataLoader bottleneck. Try CFG["num_workers"]=4')
    print('      - pin_memory=True with workers=0 helps less than expected')
    print('      - batch_size=8 might be over GPU mem with bf16; try 4')
else:
    print('   ✅ Looking good. Proceeding to training.')

Speed sanity check (10 batches, no optim step)...
   bf16 forward only: 6247 ms/batch (6.247 s/it)
   v6 baseline was ~0.5 s/it including backward — bf16 forward should be well under that
   ⚠️ Still slow. Possible issues:
      - num_workers=0 → DataLoader bottleneck. Try CFG["num_workers"]=4
      - pin_memory=True with workers=0 helps less than expected
      - batch_size=8 might be over GPU mem with bf16; try 4


In [24]:
# ── Validate loaded checkpoint baseline ───────────────────────────────────
print('\nValidating loaded v6 checkpoint baseline...')
init_vd = validate(model, val_loader, use_tta=CFG['use_tta_val'])
print(f'   Macro = {init_vd["macro_best"]:.4f}  '
      f'(per-class: {init_vd["per_class_dice"]})')
print(f'   Per-class thresholds: {init_vd["per_class_threshold"]}')
best_val_dice = init_vd['macro_best']
patience_ctr = 0; history = []


Validating loaded v6 checkpoint baseline...


   Macro = 0.8711  (per-class: {'Normal': np.float64(1.0), 'Tumor': np.float64(0.8448951677885239), 'Stone': np.float64(0.7685154837112316)})
   Per-class thresholds: {'Normal': 0.3, 'Tumor': 0.5, 'Stone': 0.5}


In [25]:
# ── Resume training in bf16 ───────────────────────────────────────────────
print('='*72)
print(f'  RESUMING v6 → v6-bf16 from epoch {CFG["resume_from_epoch"]+1}')
print(f'  Autocast: {AUTOCAST_DTYPE} | TF32: {torch.backends.cuda.matmul.allow_tf32}')
print(f'  cuDNN benchmark: {torch.backends.cudnn.benchmark} | deterministic: {torch.backends.cudnn.deterministic}')
print(f'  Starting macro: {best_val_dice:.4f}')
print('='*72)

start_ep = CFG['resume_from_epoch']
for epoch in range(start_ep, start_ep + CFG['epochs']):
    rel_ep = epoch - start_ep
    t0 = time.time()
    tl = train_one_epoch(model, train_loader, optimizer, epoch)
    scheduler.step()
    lr = optimizer.param_groups[1]['lr']
    elapsed = time.time() - t0

    rec = {'epoch': epoch+1, 'lr': lr, 'time': elapsed,
           **{f'train_{k}': v for k, v in tl.items()}}

    if (rel_ep + 1) % CFG['val_every'] == 0 or rel_ep == 0:
        vd = validate(model, val_loader, use_tta=CFG['use_tta_val'])
        rec['val_macro'] = vd['macro_best']
        for cls in CLASSES:
            rec[f'val_dice_{cls}'] = vd['per_class_dice'][cls]
            rec[f'val_thresh_{cls}'] = vd['per_class_threshold'][cls]
        per_class_str = ' '.join(
            f'{c[0]}:{vd["per_class_dice"][c]:.3f}@{vd["per_class_threshold"][c]:.2f}'
            for c in CLASSES)
        print(f'\nEpoch {epoch+1:03d} | Loss: {tl["total"]:.4f} | '
              f'Macro: {vd["macro_best"]:.4f} [{per_class_str}] | '
              f'LR: {lr:.2e} | {elapsed:.0f}s')

        if vd['macro_best'] > best_val_dice:
            best_val_dice = vd['macro_best']; patience_ctr = 0
            torch.save({'epoch': epoch+1, 'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'val': vd, 'config': CFG}, CKPT_DIR / 'best_model.pth')
            print(f'   ★ New best! Macro = {best_val_dice:.4f}')
        else:
            patience_ctr += CFG['val_every']
            if patience_ctr >= CFG['patience']:
                print(f'\n⏹ Early stopping at epoch {epoch+1}'); break
    elif (rel_ep + 1) % 10 == 0:
        print(f'Epoch {epoch+1:03d} | Loss: {tl["total"]:.4f} | LR: {lr:.2e} | {elapsed:.0f}s')

    if (rel_ep + 1) % CFG['save_every'] == 0:
        torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'config': CFG},
                   CKPT_DIR / f'checkpoint_ep{epoch+1:03d}.pth')
    history.append(rec)

torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'config': CFG},
           CKPT_DIR / 'final_model.pth')
pd.DataFrame(history).to_csv(LOG_DIR / 'training_history_bf16.csv', index=False)
print(f'\n{"="*72}\n  DONE — Best Macro: {best_val_dice:.4f}\n{"="*72}')

  RESUMING v6 → v6-bf16 from epoch 21
  Autocast: torch.bfloat16 | TF32: True
  cuDNN benchmark: False | deterministic: True
  Starting macro: 0.8711


KeyboardInterrupt: 

## Section 9 — Plots

In [ ]:
hdf = pd.read_csv(LOG_DIR / 'training_history_bf16.csv')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'v6-bf16 Resume from epoch {CFG["resume_from_epoch"]+1}', fontsize=14, fontweight='bold')

ax = axes[0,0]
ax.plot(hdf['epoch'], hdf['train_total'], alpha=0.8, label='Train Total')
if 'train_seg' in hdf.columns: ax.plot(hdf['epoch'], hdf['train_seg'], alpha=0.6, label='Seg')
if 'train_aux' in hdf.columns: ax.plot(hdf['epoch'], hdf['train_aux'], alpha=0.6, label='Stone Aux')
ax.set_title('Loss'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0,1]
vr = hdf.dropna(subset=['val_macro']) if 'val_macro' in hdf.columns else hdf
if 'val_macro' in vr.columns:
    ax.plot(vr['epoch'], vr['val_macro'], 'k-o', label='Macro', lw=2, ms=4)
    for c, col in zip(CLASSES, ['#4e79a7','#e15759','#f28e2b']):
        k = f'val_dice_{c}'
        if k in vr.columns: ax.plot(vr['epoch'], vr[k], '--', label=c, color=col)
ax.set_title('Val Dice'); ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0.5, 1.05)

ax = axes[1,0]
for comp in ['seg_focal','seg_tversky','seg_chamfer','seg_smooth']:
    k = f'train_{comp}'
    if k in hdf.columns: ax.plot(hdf['epoch'], hdf[k], label=comp, alpha=0.7)
ax.set_title('SCRL components'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1,1]
ax.plot(hdf['epoch'], hdf['lr']); ax.set_title('LR'); ax.grid(alpha=0.3); ax.set_yscale('log')

plt.tight_layout()
plt.savefig(LOG_DIR / 'training_curves_bf16.png', dpi=150, bbox_inches='tight')
plt.show()

---
### What to expect

1. **Speed sanity cell** prints `bf16 forward only: ~150-300 ms/batch`. Anything well under 1 s/it means the speed problem is fixed.
2. **Initial validation macro = 0.87** — confirms loaded weights are intact.
3. **Training epoch time ~430-500 s** — same as v6 was, which is the goal.
4. **Zero `⚠️ skipped (output NaN)`** messages — bf16 has FP32 exponent range, so no saturation.
5. **Macro climbs past 0.87** within 5-10 epochs.

### If still slow after this

- `CFG['num_workers'] = 4` (was 0 — DataLoader is single-threaded right now). Set this if the speed-test cell shows forward < 0.3 s but training is much slower.
- If GPU memory tight, drop batch to 4 — bf16 saves vs FP32 but Hiera-L is still big.

### If NaN somehow returns in bf16 (unlikely)

Means the grad clip is too lax. Try `CFG['grad_clip'] = 0.1`. If that doesn't fix it, the LoRA scaling is too aggressive — drop to `CFG['lr_lora'] = 1e-5`.